# LearnOS — Tier 1: Subject Router

**Classical ML.** A TF-IDF + Logistic Regression classifier that routes a student message to one of five agents: `dsa`, `dbms`, `maths`, `aiml`, `general`.

Why a trained classifier instead of asking the LLM every time: it's instant, free, deterministic, and gives a **confidence score** we can show in the Agent Trace panel. The LLM coordinator only gets involved when this model is unsure (below 0.6).

**Outputs** (loaded automatically by `backend/app/agents/router.py`):
- `../router/router_vectorizer.joblib`
- `../router/router_classifier.joblib`

Run every cell top to bottom. Takes well under a minute on CPU.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ROUTER_DIR = os.path.abspath(os.path.join("..", "router"))
DATA_PATH = os.path.join(ROUTER_DIR, "training_data.csv")
print("Router dir:", ROUTER_DIR)

## 1. Load the labeled data

If `training_data.csv` doesn't exist yet, generate it first:

```bash
cd ml/router && python generate_data.py
```

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Run: cd ml/router && python generate_data.py"
    )

df = pd.read_csv(DATA_PATH)
print(f"{len(df)} rows\n")
print(df["label"].value_counts())
df.sample(8, random_state=0)

## 2. Train / test split

Stratified so every class keeps its proportion in both splits. The held-out set is what you quote to judges when they ask *"did you actually train anything?"*

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)
print(f"train: {len(X_train_text)}   test: {len(X_test_text)}")

## 3. Vectorize + train

`ngram_range=(1, 2)` picks up two-word phrases like *"binary search"* or *"conditional probability"*, which carry far more signal than single words.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1,
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

clf = LogisticRegression(max_iter=1000, C=4.0, class_weight="balanced")
clf.fit(X_train, y_train)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Classes:", list(clf.classes_))

## 4. Evaluate — this is the number you quote to judges

In [ ]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Held-out accuracy: {acc:.1%}\n")
print(classification_report(y_test, y_pred))

cv = cross_val_score(clf, X_train, y_train, cv=5)
print(f"5-fold CV: {cv.mean():.1%} (+/- {cv.std():.1%})")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in clf.classes_],
                         columns=[f"pred_{c}" for c in clf.classes_])
print("Confusion matrix — off-diagonal cells are where routing would go wrong:\n")
cm_df

## 5. Inspect the confidence behavior

The threshold matters as much as the accuracy: below `0.6` the coordinator hands off to the LLM classifier instead. Check that ambiguous questions actually land below it.

In [ ]:
THRESHOLD = 0.6

probe = [
    "Why does my recursion code run forever?",
    "Write a SQL query to join these two tables",
    "Explain Bayes theorem",
    "How does gradient descent actually work?",
    "I have 2 months before placements, what should I focus on?",
    "help",
    "what should I do next",
]

probs = clf.predict_proba(vectorizer.transform(probe))
for text, row in zip(probe, probs):
    i = row.argmax()
    agent, conf = clf.classes_[i], row[i]
    route = agent if conf >= THRESHOLD else f"LLM fallback (model said {agent})"
    print(f"{conf:5.1%}  ->  {route:42s}  | {text}")

In [ ]:
test_probs = clf.predict_proba(X_test)
max_conf = test_probs.max(axis=1)
fallback_rate = (max_conf < THRESHOLD).mean()

print(f"Mean confidence on held-out set: {max_conf.mean():.1%}")
print(f"Share that would fall back to the LLM: {fallback_rate:.1%}")
print("\n(A fallback rate near zero means the threshold is doing nothing;")
print(" very high means the model is too timid. Single-digit percent is healthy.)")

## 6. What the model actually learned

Useful for the pitch — it shows the classifier keyed on real subject vocabulary rather than memorizing noise.

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())

for idx, cls in enumerate(clf.classes_):
    top = np.argsort(clf.coef_[idx])[-10:][::-1]
    print(f"{cls:8s}: {', '.join(feature_names[top])}")

## 7. Save the artifacts

`backend/app/agents/router.py` looks for exactly these two filenames in `ml/router/` and picks them up automatically on the next backend restart — no code change needed.

In [ ]:
os.makedirs(ROUTER_DIR, exist_ok=True)

vec_path = os.path.join(ROUTER_DIR, "router_vectorizer.joblib")
clf_path = os.path.join(ROUTER_DIR, "router_classifier.joblib")

joblib.dump(vectorizer, vec_path)
joblib.dump(clf, clf_path)

print("Saved:")
print(" ", vec_path)
print(" ", clf_path)
print(f"\nHeld-out accuracy for the pitch: {acc:.1%}")
print("\nRestart the backend and routing switches from keyword fallback to this model.")

In [ ]:
# Sanity check: reload from disk exactly the way the backend will
v2 = joblib.load(vec_path)
c2 = joblib.load(clf_path)

msg = "Explain conditional probability with an example"
p = c2.predict_proba(v2.transform([msg]))[0]
i = p.argmax()
print(f"{msg!r}\n  -> agent={c2.classes_[i]}  confidence={p[i]:.2f}")